# Évaluation et sortie structurée

## Présentation

Bienvenue dans ce nouveau notebook du cours sur l'IA générative. Vous allez découvrir quelques techniques pour évaluer les réponses d'un modèle de langage. Pour cette évaluation, vous utiliserez aussi la capacité de Gemini à produire des **données structurées**, afin d'obtenir les résultats sous forme d'objets Python.

Remarque : ce notebook contient plus de code que le premier notebook du jour 1 (Prompting). Il n'est pas indispensable pour la suite du cours : vous pouvez le passer ou y revenir plus tard dans la semaine. Si vous n'avez pas encore fait le notebook Prompting, commencez par celui-là : il présente les bases de l'interaction avec les LLM.

Consultez aussi le **livre blanc bonus** [Evaluating Large Language Models](https://services.google.com/fh/files/blogs/neurips_evaluation.pdf) (en anglais).

> **Version adaptée pour Google Colab** : ce notebook est en français, lit la clé API depuis les **secrets Colab**, relance automatiquement les appels en cas de quota atteint et utilise le modèle `gemini-3.1-flash-lite` (les modèles Gemini 2.0 d'origine ont été arrêtés). Les prompts d'évaluation restent en anglais : ce sont des modèles de prompts officiels de Google Cloud.

## Configuration

Installez le SDK Python.

In [1]:
# Étape 1 : Mise à jour du SDK Google GenAI vers la version la plus récente
# Cela est indispensable pour pouvoir utiliser le modèle récent 'gemini-3.1-flash-lite'
!pip install -U -q "google-genai"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 13.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.61.0 which is incompatible.


In [2]:
# Étape 2 : Importation des bibliothèques nécessaires
from google import genai
from google.genai import types
from IPython.display import Markdown, display

# Affichage de la version installée pour vérification
print("Version du SDK Google GenAI:", genai.__version__)

# Définition du modèle par défaut utilisé tout au long de ce notebook
MODEL = "gemini-3.1-flash-lite"

### Configurer votre clé API

Pour exécuter la cellule suivante, votre clé API doit être enregistrée dans un [secret Colab](https://colab.research.google.com/notebooks/secrets.ipynb) nommé `GOOGLE_API_KEY` :

1. Dans la barre latérale gauche de Colab, cliquez sur l'icône **clé** 🔑 (« Secrets »).
2. Cliquez sur **Ajouter un secret**, saisissez `GOOGLE_API_KEY` comme nom et collez votre clé comme valeur.
3. Activez l'interrupteur **Accès au notebook**.

Si vous n'avez pas encore de clé API, vous pouvez en obtenir une sur [AI Studio](https://aistudio.google.com/app/apikey). Vous trouverez des [instructions détaillées dans la documentation](https://ai.google.dev/gemini-api/docs/api-key).

Hors de Colab (exécution locale, VM…), la cellule utilise la variable d'environnement `GOOGLE_API_KEY`.

In [17]:
import os

# Initialisation de la clé API par défaut
GOOGLE_API_KEY = "GOOGLE_API_KEY"

# 1) Tentative de récupération depuis les secrets de Google Colab
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass  # Si nous ne sommes pas dans un environnement Colab, ignore l'erreur

# 2) Alternative : Récupération depuis les variables d'environnement locales
if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

# 3) Sécurité : Interruption si aucune clé n'est configurée
if not GOOGLE_API_KEY:
    raise ValueError(
        "Clé API introuvable : ajoutez un secret Colab nommé GOOGLE_API_KEY "
        "et activez « Accès au notebook »."
    )

# Initialisation du client GenAI officiel avec la clé API
client = genai.Client(api_key=GOOGLE_API_KEY)
print("Clé API chargée avec succès ✅")

Clé API chargée avec succès ✅


Si vous obtenez une erreur du type `SecretNotFoundError` ou `NotebookAccessError`, vérifiez que le secret s'appelle exactement `GOOGLE_API_KEY` **et** que l'interrupteur **Accès au notebook** est activé dans le panneau Secrets (icône clé). Colab peut aussi afficher une fenêtre vous demandant d'autoriser l'accès : cliquez sur **Autoriser l'accès**.

### Nouvelle tentative automatique (retry)

Ce notebook envoie beaucoup de requêtes : mettez en place une nouvelle tentative automatique quand le quota par minute est atteint. En cas d'erreur 429 (quota) ou 503 (modèle surchargé), l'appel est relancé automatiquement après 2 s, 4 s, 8 s… pendant 60 s au maximum. Il n'y a aucune attente quand l'appel réussit.

> **Clé gratuite** : le tier gratuit est partagé et peut être saturé pendant le cours. Si une cellule met un peu de temps à répondre, c'est normal : laissez le retry faire son travail. Si l'erreur persiste après 60 s, relancez simplement la cellule.

In [4]:
from google.api_core import retry

# Définition d'un filtre pour identifier les erreurs qui justifient une nouvelle tentative
# Code 429 = Trop de requêtes (quota) | Code 503 = Service temporairement indisponible (surcharge)
is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})

# Configuration de la politique de retry (attente progressive de 2s à 15s max, arrêt après 60s cumulées)
politique_retry = retry.Retry(predicate=is_retriable, initial=2, maximum=15, timeout=60)

# Sauvegarde de la méthode d'origine pour éviter une surcharge infinie si la cellule est réexécutée
if not hasattr(genai.models.Models, "_generate_original"):
    genai.models.Models._generate_original = genai.models.Models.generate_content

# Application de la politique de retry automatique sur la fonction principale de génération
genai.models.Models.generate_content = politique_retry(genai.models.Models._generate_original)

## Évaluation

Quand on utilise des LLM dans des cas réels, il est important de savoir s'ils fonctionnent bien. Comme les LLM génèrent du texte libre, beaucoup de cas sont difficiles à mesurer. Dans ce notebook, vous allez parcourir quelques techniques simples pour évaluer les réponses d'un LLM et comprendre ses performances.

Dans cet exemple, vous allez évaluer une tâche de résumé à partir du [rapport technique de Gemini 1.5 Pro](https://storage.googleapis.com/cloud-samples-data/generative-ai/pdf/2403.05530.pdf). Commencez par télécharger le PDF dans l'environnement du notebook, puis envoyez cette copie à l'API Gemini.

In [5]:
# Téléchargement du PDF du rapport technique Gemini 1.5 depuis l'URL de test
!wget -nv -O gemini.pdf https://storage.googleapis.com/cloud-samples-data/generative-ai/pdf/2403.05530.pdf

# Importation du fichier dans l'API Gemini Files pour pouvoir le passer en contexte au modèle
document_file = client.files.upload(file='gemini.pdf')
print("Fichier téléchargé et envoyé à l'API Gemini avec succès 📄")

2026-10-07 21:58:17 URL:https://storage.googleapis.com/cloud-samples-data/generative-ai/pdf/2403.05530.pdf [7228817/7228817] -> "gemini.pdf" [1]


### Résumer un document

La demande de résumé utilisée ici est assez basique : elle cible le processus d'entraînement, sans autre consigne.

In [7]:
# Définition de la consigne de résumé à envoyer au modèle
request = 'Tell me about the training process used here.'

def summarise_doc(request: str) -> str:
  """Exécute la demande de résumé sur le document PDF fourni."""
  # Utilisation d'une température à 0.0 pour un résultat déterministe et stable
  config = types.GenerateContentConfig(temperature=0.0)

  # Appel de l'API en combinant la question textuelle et le document PDF
  response = client.models.generate_content(
      model=MODEL,
      config=config,
      contents=[request, document_file],
  )

  return response.text

# Génération et affichage du résumé initial
summary = summarise_doc(request)
Markdown(summary)

Based on the provided report, the training process for Gemini 1.5 Pro involves the following key components:

*   **Infrastructure:** The model is trained on multiple 4096-chip pods of Google’s TPUv4 accelerators, which are distributed across multiple datacenters.
*   **Architecture:** It is a sparse mixture-of-experts (MoE) Transformer-based model. This architecture uses a learned routing function to direct inputs to a subset of the model's parameters, allowing the model to grow its total parameter count while keeping the number of activated parameters constant for any given input.
*   **Data:** The pre-training dataset includes a variety of multimodal and multilingual data sourced from many different domains, including web documents, code, image, audio, and video content.
*   **Instruction Tuning:** Following pre-training, the model undergoes an instruction-tuning phase where it is fine-tuned on a collection of multimodal data containing paired instructions and responses. This is further refined using human preference data.
*   **Software/Frameworks:** Training is conducted using JAX and ML Pathways. JAX, powered by XLA, includes the GSPMD partitioner for automatic parallelization, which allows for the orchestration of the training run using a "single controller" programming model.

### Définir un évaluateur

Pour une tâche comme celle-ci, vous pouvez évaluer plusieurs aspects : la façon dont le modèle a suivi le prompt (« respect des instructions »), le fait qu'il s'appuie sur les données fournies (« ancrage », *groundedness*), la facilité de lecture du texte (« fluidité »), ou d'autres critères comme la « verbosité » ou la « qualité ».

Vous pouvez demander à un LLM de réaliser cette évaluation comme vous le feriez avec un évaluateur humain : avec une définition claire et une [grille d'évaluation](https://fr.wikipedia.org/wiki/Grille_d%27%C3%A9valuation).

Dans cette étape, vous définissez un agent d'évaluation à partir d'un prompt de « résumé » déjà rédigé, et vous l'utilisez pour mesurer la qualité du résumé généré.

Remarque : pour d'autres prompts d'évaluation prêts à l'emploi (ancrage, sécurité, cohérence, etc.), consultez cette [liste complète de prompts d'évaluation par modèle](https://cloud.google.com/vertex-ai/generative-ai/docs/models/metrics-templates) dans la documentation Google Cloud.

In [8]:
import enum

# Modèle de prompt officiel (Google Cloud) guidant le LLM pour agir en tant qu'expert évaluateur
SUMMARY_PROMPT = """\
# Instruction
You are an expert evaluator. Your task is to evaluate the quality of the responses generated by AI models.
We will provide you with the user input and an AI-generated responses.
You should first read the user input carefully for analyzing the task, and then evaluate the quality of the responses based on the Criteria provided in the Evaluation section below.
You will assign the response a rating following the Rating Rubric and Evaluation Steps. Give step-by-step explanations for your rating, and only choose ratings from the Rating Rubric.

# Evaluation
## Metric Definition
You will be assessing summarization quality, which measures the overall ability to summarize text. Pay special attention to length constraints, such as in X words or in Y sentences. The instruction for performing a summarization task and the context to be summarized are provided in the user prompt. The response should be shorter than the text in the context. The response should not contain information that is not present in the context.

## Criteria
Instruction following: The response demonstrates a clear understanding of the summarization task instructions, satisfying all of the instruction's requirements.
Groundedness: The response contains information included only in the context. The response does not reference any outside information.
Conciseness: The response summarizes the relevant details in the original text without a significant loss in key information without being too verbose or terse.
Fluency: The response is well-organized and easy to read.

## Rating Rubric
5: (Very good). The summary follows instructions, is grounded, is concise, and fluent.
4: (Good). The summary follows instructions, is grounded, concise, and fluent.
3: (Ok). The summary mostly follows instructions, is grounded, but is not very concise and is not fluent.
2: (Bad). The summary is grounded, but does not follow the instructions.
1: (Very bad). The summary is not grounded.

## Evaluation Steps
STEP 1: Assess the response in aspects of instruction following, groundedness, conciseness, and verbosity according to the criteria.
STEP 2: Score based on the rubric.

# User Inputs and AI-generated Response
## User Inputs

### Prompt
{prompt}

## AI-generated Response
{response}
"""

# Définition de l'énumération Python pour contraindre la sortie structurée à une note de 1 à 5
class SummaryRating(enum.Enum):
  VERY_GOOD = '5'
  GOOD = '4'
  OK = '3'
  BAD = '2'
  VERY_BAD = '1'


def eval_summary(prompt, ai_response):
  """Évalue le résumé généré à l'aide d'un chat conversationnel (historique partagé)."""
  chat = client.chats.create(model=MODEL)

  # 1er tour : Demande de justification textuelle détaillée (Step-by-step)
  response = chat.send_message(
      message=SUMMARY_PROMPT.format(prompt=prompt, response=ai_response)
  )
  verbose_eval = response.text

  # 2ème tour : Demande de conversion finale en un format structuré stricte (Enum)
  structured_output_config = types.GenerateContentConfig(
      response_mime_type="text/x.enum",
      response_schema=SummaryRating,
  )
  response = chat.send_message(
      message="Convert the final score.",
      config=structured_output_config,
  )
  structured_eval = response.parsed

  return verbose_eval, structured_eval

# Lancement du processus d'évaluation sur le résumé généré précédemment
text_eval, struct_eval = eval_summary(prompt=[request, document_file], ai_response=summary)
Markdown(text_eval)

To evaluate the AI-generated response, I have analyzed the content against the provided criteria.

**STEP 1: Assessment**
*   **Instruction Following:** The user requested an explanation of the training process based on the uploaded file (the Gemini 1.5 technical report). The model successfully extracted the relevant technical details and formatted them clearly.
*   **Groundedness:** The information provided (TPUv4 pods, sparse MoE architecture, JAX/ML Pathways, and the pre-training/instruction-tuning phases) is consistent with the publicly available technical report for Gemini 1.5 Pro. No outside information was introduced that contradicts the source material.
*   **Conciseness:** The response is highly effective at summarizing a complex technical document into five distinct, high-impact bullet points. It avoids unnecessary fluff while retaining the critical components of the training pipeline.
*   **Fluency:** The language is professional, well-structured, and easy to read.

**STEP 2: Scoring**
The response demonstrates a sophisticated understanding of the source material. It correctly identifies the core pillars of the training process—infrastructure, architecture, data, tuning, and software—and communicates them clearly. It meets all criteria for a high-quality summary.

**Rating: 5**

Dans cet exemple, le modèle a produit une justification textuelle dans le cadre d'une conversation (chat). Cette réponse complète est utile à la fois pour la lecture humaine et pour donner au modèle un espace où « prendre des notes » pendant qu'il évalue le texte et produit la note finale. Cette stratégie de « prise de notes » ou de « réflexion » fonctionne généralement bien avec les modèles autorégressifs, où le texte généré est réinjecté dans le modèle à chaque étape de génération : les « notes » sont donc utilisées pour produire le résultat final.

Au tour suivant, le modèle convertit sa réponse textuelle en réponse structurée. Si vous voulez agréger des notes ou les utiliser dans du code, mieux vaut éviter d'analyser du texte libre. Ici, on passe le schéma `SummaryRating` : le modèle convertit l'historique de la conversation en une instance de l'énumération `SummaryRating`.

In [9]:
# Affichage de la note finale structurée (issue de l'énumération Python SummaryRating)
print("Note structurée obtenue :", struct_eval)

Note structurée obtenue : SummaryRating.VERY_GOOD


### Améliorer ou dégrader le prompt de résumé

Les modèles Gemini sont en général assez bons pour résumer directement, sans beaucoup de consignes : attendez-vous à un résultat `GOOD` ou `VERY_GOOD` sur la tâche précédente, même avec un prompt rudimentaire. Exécutez-la plusieurs fois pour vous faire une idée de la réponse moyenne.

Pour voir comment influencer le résumé, réfléchissez à ce que vous pourriez changer dans la demande. Le prompt d'évaluation `SUMMARY_PROMPT` peut vous donner des idées.

Essayez les modifications suivantes et observez leur effet, positif ou négatif, sur le résultat :
* Précisez la longueur du résumé,
* Demandez des informations précises,
* Demandez des informations absentes du document,
* Demandez différents niveaux de résumé (par exemple « explique comme à un enfant de 5 ans » ou « avec tous les détails techniques »).

In [10]:
# Définition d'un nouveau prompt pour tester la sensibilité et la qualité de l'évaluation
new_prompt = "Explain like I'm 5 the training process"

if not new_prompt:
  raise ValueError("Définissez un nouveau prompt de résumé.")


def run_and_eval_summary(prompt):
  """Génère un nouveau résumé puis lance l'évaluateur automatique pour analyser le résultat."""
  # Génération du résumé simplifié (ELI5)
  summary = summarise_doc(new_prompt)
  display(Markdown("### Résumé généré :\n" + summary + '\n-----'))

  # Évaluation du résumé simplifié par rapport aux critères définis
  text, struct = eval_summary([new_prompt, document_file], summary)
  display(Markdown("### Évaluation détaillée :\n" + text + '\n-----'))
  print("Note d'évaluation structurée :", struct)

run_and_eval_summary(new_prompt)

### Résumé généré :
Imagine you have a giant library of books, movies, and audio recordings. You want to teach a robot to understand everything in that library so it can answer any question you ask about it.

Here is how the training process works, explained simply:

**1. The "Big Reading" Phase (Pre-training)**
First, the robot is given a massive amount of information—books, websites, videos, and audio files. It doesn't just "read" them; it plays a game of "guess the next piece." It looks at a sentence or a video frame and tries to predict what comes next. By doing this billions of times, it learns how language works, how stories are structured, and how to recognize patterns in pictures and sounds.

**2. The "Teacher" Phase (Instruction Tuning)**
After the robot has learned the basics, it needs to learn how to be helpful. In this step, human teachers give it specific examples of questions and the "perfect" answers. For example, they might show it a long document and ask, "What is the main character's name?" The robot learns that when it sees a question, it should look through the information it has to find the right answer, rather than just guessing.

**3. The "Practice" Phase (Fine-tuning)**
The robot is then tested on harder tasks, like learning a brand-new language just by reading a grammar book, or finding a specific moment in a 10-hour movie. This helps the robot get really good at "long-context" tasks—meaning it can remember things from the very beginning of a huge amount of information, even if that information is hours long or thousands of pages thick.

**4. The "Safety" Phase (Making it Safe)**
Finally, humans check to make sure the robot isn't saying anything mean, dangerous, or wrong. They give it "adversarial" tests—basically trying to trick it into saying something bad—and then teach it to refuse those requests or provide safe, helpful answers instead.

**Why is this special?**
Most robots can only remember a small amount of information at once (like a short story). Gemini 1.5 Pro is special because it has been trained to have a "super-memory," allowing it to hold millions of pieces of information in its "head" at the same time, so it can answer questions about huge collections of data that other robots would forget.
-----

### Évaluation détaillée :
### Evaluation

**STEP 1: Assess the response in aspects of instruction following, groundedness, conciseness, and verbosity according to the criteria.**

*   **Instruction Following:** The user requested an explanation of the training process "like I'm 5". The model uses an excellent analogy of a robot learning from a library, which is appropriate for the target audience. It correctly breaks down the components of the training process (Pre-training, Instruction Tuning, Fine-tuning/Long-context, and Safety) based on the provided document (Gemini 1.5 Pro technical documentation).
*   **Groundedness:** The information provided aligns with the Gemini 1.5 Pro technical report (the source file). The explanation of the phases—pre-training, instruction tuning, long-context handling, and safety (RLHF/adversarial testing)—is accurately derived from the context of how this specific model was developed.
*   **Conciseness:** The response is well-structured and concise. It uses headings and simple language to explain complex technical concepts without getting bogged down in jargon, successfully summarizing a dense technical document.
*   **Fluency:** The tone is warm, engaging, and highly readable. It flows logically from the initial concept to the specific capabilities of the model.

**STEP 2: Score based on the rubric.**

The response is a perfect example of a summary that adheres to all constraints. It explains complex technical concepts in a way a child could understand while remaining strictly grounded in the provided source material.

**Rating: 5**
-----

Note d'évaluation structurée : SummaryRating.VERY_GOOD


## L'évaluation en pratique

L'évaluation a de nombreux usages concrets, par exemple :
* Itérer rapidement sur un prompt avec un petit ensemble de documents de test,
* Comparer différents modèles pour trouver celui qui convient le mieux, par exemple le meilleur compromis prix/performance, ou la meilleure performance sur une tâche précise,
* Vérifier qu'un changement de modèle ou de prompt dans un système en production ne dégrade pas la qualité (non-régression).

Dans cette section, vous allez essayer deux approches d'évaluation différentes.

### Évaluation ponctuelle (pointwise)

La technique utilisée plus haut, qui consiste à évaluer une seule paire entrée/sortie selon des critères, s'appelle l'évaluation ponctuelle (*pointwise*). Elle sert à juger une réponse de façon absolue : « est-ce bon ou mauvais ? ».

Dans cet exercice, vous allez tester différents prompts de consigne sur un ensemble de questions.

In [11]:
import functools

# Définition de différentes consignes (guidelines) pour tester l'impact du ton et de la précision
terse_guidance = "Answer the following question in a single sentence, or as close to that as possible."
moderate_guidance = "Provide a brief answer to the following question, use a citation if necessary, but only enough to answer the question."
cited_guidance = "Provide a thorough, detailed answer to the following question, citing the document and supplying additional background information as much as possible."

guidance_options = {
    'Terse': terse_guidance,
    'Moderate': moderate_guidance,
    'Cited': cited_guidance,
}

# Choix de la question d'évaluation
questions = [
    "How does the model perform on code tasks?",
]

if not questions:
  raise NotImplementedError('Ajoutez des questions à évaluer.')


# Utilisation de @functools.cache pour éviter de faire des appels API répétés si les paramètres sont identiques
@functools.cache
def answer_question(question: str, guidance: str = '') -> str:
  """Répond à la question à partir du document PDF et des instructions système."""
  config = types.GenerateContentConfig(
      temperature=0.0,
      system_instruction=guidance,
  )
  response = client.models.generate_content(
      model=MODEL,
      config=config,
      contents=[question, document_file],
  )

  return response.text

# Génération d'une réponse de test avec l'instruction concise (Terse)
answer = answer_question(questions[0], terse_guidance)
Markdown(answer)

Gemini 1.5 Pro demonstrates strong performance on code tasks, as evidenced by its ability to ingest and answer specific queries about large codebases like JAX, and its achievement of an 8.9% improvement in coding benchmarks compared to Gemini 1.0 Pro, while also surpassing Gemini 1.0 Ultra on the internal Natural2Code benchmark.

Mettez maintenant en place un évaluateur de questions-réponses, comme précédemment, mais avec le [prompt d'évaluation ponctuelle QA](https://cloud.google.com/vertex-ai/generative-ai/docs/models/metrics-templates#pointwise_question_answering_quality).

In [12]:
import enum

# Modèle de prompt officiel (Google Cloud) pour l'évaluation ponctuelle (Pointwise QA)
QA_PROMPT = """\
# Instruction
You are an expert evaluator. Your task is to evaluate the quality of the responses generated by AI models.
We will provide you with the user prompt and an AI-generated responses.
You should first read the user prompt carefully for analyzing the task, and then evaluate the quality of the responses based on and rules provided in the Evaluation section below.

You will assign the writing response a score from 5, 4, 3, 2, 1, following the Rating Rubric and Evaluation Steps.
Give step-by-step explanations for your scoring, and only choose scores from 5, 4, 3, 2, 1.

# Evaluation
## Metric Definition
You will be assessing question answering quality, which measures the overall quality of the answer to the question in the user prompt. Pay special attention to length constraints, such as in X words or in Y sentences. The instruction for performing a question-answering task is provided in the user prompt. The response should not contain information that is not present in the context (if it is provided).

You will assign the writing response a score from 5, 4, 3, 2, 1, following the Rating Rubric and Evaluation Steps.
Give step-by-step explanations for your scoring, and only choose scores from 5, 4, 3, 2, 1.

## Criteria Definition
Instruction following: The response demonstrates a clear understanding of the question answering task instructions, satisfying all of the instruction's requirements.
Groundedness: The response contains information included only in the context if the context is present in the user prompt. The response does not reference any outside information.
Completeness: The response completely answers the question with sufficient detail.
Fluent: The response is well-organized and easy to read.

## Rating Rubric
5: (Very good). The answer follows instructions, is grounded, complete, and fluent.
4: (Good). The answer follows instructions, is grounded, complete, but is not very fluent.
3: (Ok). The answer mostly follows instructions, is grounded, answers the question partially and is not very fluent.
2: (Bad). The answer does not follow the instructions very well, is incomplete or not fully grounded.
1: (Very bad). The answer does not follow the instructions, is wrong and not grounded.

## Evaluation Steps
STEP 1: Assess the response in aspects of instruction following, groundedness,completeness, and fluency according to the criteria.
STEP 2: Score based on the rubric.

# User Inputs and AI-generated Response
## User Inputs
### Prompt
{prompt}

## AI-generated Response
{response}
"""

# Énumération pour contraindre le score final de l'évaluation QA
class AnswerRating(enum.Enum):
  VERY_GOOD = '5'
  GOOD = '4'
  OK = '3'
  BAD = '2'
  VERY_BAD = '1'


@functools.cache
def eval_answer(prompt, ai_response, n=1):
  """Évalue une réponse spécifique par rapport à la question posée."""
  chat = client.chats.create(model=MODEL)

  # Étape 1 : Justification détaillée
  response = chat.send_message(
      message=QA_PROMPT.format(prompt=[prompt, document_file], response=ai_response)
  )
  verbose_eval = response.text

  # Étape 2 : Conversion du score final en type structuré
  structured_output_config = types.GenerateContentConfig(
      response_mime_type="text/x.enum",
      response_schema=AnswerRating,
  )
  response = chat.send_message(
      message="Convert the final score.",
      config=structured_output_config,
  )
  structured_eval = response.parsed

  return verbose_eval, structured_eval

# Évaluation de la réponse obtenue précédemment	ext_eval, struct_eval = eval_answer(prompt=questions[0], ai_response=answer)
display(Markdown(text_eval))
print("Note de la réponse :", struct_eval)

To evaluate the AI-generated response, I have analyzed the content against the provided criteria.

**STEP 1: Assessment**
*   **Instruction Following:** The user requested an explanation of the training process based on the uploaded file (the Gemini 1.5 technical report). The model successfully extracted the relevant technical details and formatted them clearly.
*   **Groundedness:** The information provided (TPUv4 pods, sparse MoE architecture, JAX/ML Pathways, and the pre-training/instruction-tuning phases) is consistent with the publicly available technical report for Gemini 1.5 Pro. No outside information was introduced that contradicts the source material.
*   **Conciseness:** The response is highly effective at summarizing a complex technical document into five distinct, high-impact bullet points. It avoids unnecessary fluff while retaining the critical components of the training pipeline.
*   **Fluency:** The language is professional, well-structured, and easy to read.

**STEP 2: Scoring**
The response demonstrates a sophisticated understanding of the source material. It correctly identifies the core pillars of the training process—infrastructure, architecture, data, tuning, and software—and communicates them clearly. It meets all criteria for a high-quality summary.

**Rating: 5**

Note de la réponse : SummaryRating.VERY_GOOD


Exécutez maintenant l'évaluation en boucle. Notez que la consigne est cachée à l'agent d'évaluation. Si vous la lui transmettiez, il noterait le respect de cette consigne. Or, ici, l'objectif est de trouver le meilleur résultat global par rapport à la question de l'utilisateur, et non par rapport à l'instruction du développeur.

In [15]:
import collections
import itertools

# Nombre de fois que chaque évaluation est répétée pour limiter les écarts statistiques
NUM_ITERATIONS = 1

scores = collections.defaultdict(int)
responses = collections.defaultdict(list)

# Évaluation croisée systématique pour chaque question et chaque style de consigne
for question in questions:
  display(Markdown(f'## {question}'))
  for guidance, guide_prompt in guidance_options.items():

    for n in range(NUM_ITERATIONS):
      # 1. Génération de la réponse du modèle
      answer = answer_question(question, guide_prompt)

      # 2. Évaluation de la qualité de cette réponse (la consigne initiale n'est volontairement pas passée à l'évaluateur)
      written_eval, struct_eval = eval_answer(question, answer, n)
      print(f'Consigne : {guidance} -> Score obtenu : {struct_eval}')

      # 3. Stockage du score cumulé pour analyse statistique future
      scores[guidance] += int(struct_eval.value)

      # 4. Enregistrement de l'historique complet pour consultation manuelle
      responses[(guidance, question)].append((answer, written_eval))

## How does the model perform on code tasks?

Consigne : Terse -> Score obtenu : AnswerRating.VERY_GOOD
Consigne : Moderate -> Score obtenu : AnswerRating.VERY_GOOD
Consigne : Cited -> Score obtenu : AnswerRating.VERY_GOOD


Agrégez maintenant les notes pour voir les performances de chaque prompt.

In [18]:
# Calcul de la moyenne des notes obtenues par chaque type d'instruction de génération
for guidance, score in scores.items():
  avg_score = score / (NUM_ITERATIONS * len(questions))
  nearest = AnswerRating(str(round(avg_score)))
  print(f'{guidance}: Moyenne = {avg_score:.2f} - Équivalent le plus proche : {nearest.name}')

Terse: Moyenne = 5.00 - Équivalent le plus proche : VERY_GOOD
Moderate: Moyenne = 5.00 - Équivalent le plus proche : VERY_GOOD
Cited: Moyenne = 5.00 - Équivalent le plus proche : VERY_GOOD


### Évaluation par paires (pairwise)

Le prompt d'évaluation ponctuelle utilisé à l'étape précédente comporte 5 niveaux de note. Cela peut être trop grossier pour votre système, ou vous voulez peut-être améliorer un prompt déjà « très bon ».

Une autre approche consiste à comparer deux réponses entre elles. C'est l'évaluation par paires (*pairwise*), une étape clé des algorithmes de classement et de tri : elle permet de classer vos prompts, à la place de l'approche ponctuelle ou en complément.

Cette étape met en œuvre l'évaluation par paires avec le [prompt de qualité QA par paires](https://cloud.google.com/vertex-ai/generative-ai/docs/models/metrics-templates#pairwise_question_answering_quality) de la documentation Google Cloud.

In [19]:
# Modèle de prompt officiel (Google Cloud) pour l'évaluation comparative par paires (Pairwise QA)
QA_PAIRWISE_PROMPT = """\
# Instruction
You are an expert evaluator. Your task is to evaluate the quality of the responses generated by two AI models. We will provide you with the user input and a pair of AI-generated responses (Response A and Response B). You should first read the user input carefully for analyzing the task, and then evaluate the quality of the responses based on the Criteria provided in the Evaluation section below.

You will first judge responses individually, following the Rating Rubric and Evaluation Steps. Then you will give step-by-step explanations for your judgment, compare results to declare the winner based on the Rating Rubric and Evaluation Steps.

# Evaluation
## Metric Definition
You will be assessing question answering quality, which measures the overall quality of the answer to the question in the user prompt. Pay special attention to length constraints, such as in X words or in Y sentences. The instruction for performing a question-answering task is provided in the user prompt. The response should not contain information that is not present in the context (if it is provided).

## Criteria
Instruction following: The response demonstrates a clear understanding of the question answering task instructions, satisfying all of the instruction's requirements.
Groundedness: The response contains information included only in the context if the context is present in the user prompt. The response does not reference any outside information.
Completeness: The response completely answers the question with sufficient detail.
Fluent: The response is well-organized and easy to read.

## Rating Rubric
"A": Response A answers the given question as per the criteria better than response B.
"SAME": Response A and B answers the given question equally well as per the criteria.
"B": Response B answers the given question as per the criteria better than response A.

## Evaluation Steps
STEP 1: Analyze Response A based on the question answering quality criteria: Determine how well Response A fulfills the user requirements, is grounded in the context, is complete and fluent, and provides assessment according to the criterion.
STEP 2: Analyze Response B based on the question answering quality criteria: Determine how well Response B fulfills the user requirements, is grounded in the context, is complete and fluent, and provides assessment according to the criterion.
STEP 3: Compare the overall performance of Response A and Response B based on your analyses and assessment.
STEP 4: Output your preference of "A", "SAME" or "B" to the pairwise_choice field according to the Rating Rubric.
STEP 5: Output your assessment reasoning in the explanation field.

# User Inputs and AI-generated Responses
## User Inputs
### Prompt
{prompt}

# AI-generated Response

### Response A
{baseline_model_response}

### Response B
{response}
"""

# Structure de décision comparative stricte
class AnswerComparison(enum.Enum):
  A = 'A'
  SAME = 'SAME'
  B = 'B'


@functools.cache
def eval_pairwise(prompt, response_a, response_b, n=1):
  """Compare directement deux réponses et désigne la meilleure (A, B ou identique)."""

  chat = client.chats.create(model=MODEL)

  # Tour 1 : Analyse comparative textuelle détaillée
  response = chat.send_message(
      message=QA_PAIRWISE_PROMPT.format(
          prompt=[prompt, document_file],
          baseline_model_response=response_a,
          response=response_b)
  )
  verbose_eval = response.text

  # Tour 2 : Extraction structurée de la décision
  structured_output_config = types.GenerateContentConfig(
      response_mime_type="text/x.enum",
      response_schema=AnswerComparison,
  )
  response = chat.send_message(
      message="Convert the final score.",
      config=structured_output_config,
  )
  structured_eval = response.parsed

  return verbose_eval, structured_eval

# Exemple pratique : Comparer la version concise (A) avec la version détaillée et citée (B)
question = questions[0]
answer_a = answer_question(question, terse_guidance)
answer_b = answer_question(question, cited_guidance)

text_eval, struct_eval = eval_pairwise(
    prompt=question,
    response_a=answer_a,
    response_b=answer_b,
)

display(Markdown(text_eval))
print("Résultat du duel (pairwise) :", struct_eval)

**STEP 1: Analyze Response A**
Response A provides a concise summary of the model's performance on code tasks based on the provided technical report. It captures key metrics and specific examples mentioned in the document.

**STEP 2: Analyze Response B**
Response B provides a much more comprehensive and structured answer. It breaks down the information into specific categories (benchmarks, long-context capabilities, and evaluation methodology), cites specific tables and pages from the report, and provides much more granular data points (e.g., specific pass rates for HumanEval, NLL performance, and details on evaluation methodology).

**STEP 3: Compare results**
While Response A is accurate, Response B is significantly more helpful and thorough. It leverages the provided document to give a detailed, evidence-based answer that cites the source material (tables/pages), making it a much higher-quality answer for a user seeking information from a technical document.

**STEP 4: Output preference**
B

**STEP 5: Explanation**
Response B is significantly better because it provides a thorough and well-structured analysis of the model's code performance. It includes specific data points, references to the exact tables and pages in the source document, and explains the context of the benchmarks and the evaluation methodology. Response A is a bit too brief and misses much of the valuable detail provided in the report.

Résultat du duel (pairwise) : AnswerComparison.B


Une fois l'évaluateur par paires en place, il suffit d'un comparateur pour classer les prompts entre eux.

Cet exemple implémente les comparateurs minimaux nécessaires à un ordre total (`==` et `<`) et effectue la comparaison avec `n_iterations` évaluations sur l'ensemble des `questions`.

In [20]:
# Décorateur functools.total_ordering pour générer automatiquement tous les opérateurs de comparaison (<, >, <=, >=) à partir de __eq__ et __lt__
@functools.total_ordering
class QAGuidancePrompt:
  """Encapsule un prompt de consigne de génération de questions-réponses pour permettre son tri relatif."""

  def __init__(self, prompt, questions, n_comparisons=NUM_ITERATIONS):
    """Initialise le prompt avec les questions de test et le nombre de duels à réaliser par question."""
    self.prompt = prompt
    self.questions = questions
    self.n = n_comparisons

  def __str__(self):
    return self.prompt

  def _compare_all(self, other):
    """Compare globalement ce prompt à un autre (other) sur l'ensemble des questions d'évaluation."""
    results = [self._compare_n(other, q) for q in questions]
    mean = sum(results) / len(results)
    return round(mean)

  def _compare_n(self, other, question):
    """Compare deux prompts sur une question donnée, répétée n fois."""
    results = [self._compare(other, question, n) for n in range(self.n)]
    mean = sum(results) / len(results)
    return mean

  def _compare(self, other, question, n=1):
    """Effectue un seul duel direct entre la réponse générée par self et celle de other."""
    answer_a = answer_question(question, self.prompt)
    answer_b = answer_question(question, other.prompt)

    _, result = eval_pairwise(
        prompt=question,
        response_a=answer_a,
        response_b=answer_b,
        n=n,  # L'utilisation de l'indice n permet de contourner le cache si nécessaire
    )

    # Attribution des points pour le tri de Python :
    # Si A gagne -> +1 | Si B gagne -> -1 | Égalité -> 0
    if result is AnswerComparison.A:
      return 1
    elif result is AnswerComparison.B:
      return -1
    else:
      return 0

  def __eq__(self, other):
    """Définit la règle d'égalité de tri par évaluation par paires."""
    if not isinstance(other, QAGuidancePrompt):
      return NotImplemented

    return self._compare_all(other) == 0

  def __lt__(self, other):
    """Définit la relation 'inférieur à' (moins bon que) par évaluation par paires."""
    if not isinstance(other, QAGuidancePrompt):
      return NotImplemented

    return self._compare_all(other) < 0

Les fonctions de tri de Python fonctionnent maintenant directement sur les instances de `QAGuidancePrompt`. Les fonctions `answer_question` et `eval_pairwise` sont [mémoïsées](https://fr.wikipedia.org/wiki/M%C3%A9mo%C3%AFsation) pour éviter de régénérer inutilement les mêmes réponses ou évaluations : cette étape devrait donc être rapide, sauf si vous avez modifié les questions, les prompts ou le nombre d'itérations.

In [21]:
# Instanciation de nos prompts de consigne à l'aide de la classe QAGuidancePrompt pour le tri comparatif
terse_prompt = QAGuidancePrompt(terse_guidance, questions)
moderate_prompt = QAGuidancePrompt(moderate_guidance, questions)
cited_prompt = QAGuidancePrompt(cited_guidance, questions)

# Tri automatique des prompts de consigne en ordre descendant (du meilleur au moins bon)
sorted_results = sorted([terse_prompt, moderate_prompt, cited_prompt], reverse=True)

# Affichage du classement final
for i, p in enumerate(sorted_results):
  if i:
    print('---')
  print(f'Classement #{i+1} : {p}')

Classement #1 : Provide a thorough, detailed answer to the following question, citing the document and supplying additional background information as much as possible.
---
Classement #2 : Provide a brief answer to the following question, use a citation if necessary, but only enough to answer the question.
---
Classement #3 : Answer the following question in a single sentence, or as close to that as possible.


## Difficultés

### Limites des LLM

Les LLM ont des difficultés connues sur certaines tâches, et ces difficultés persistent quand on les utilise comme évaluateurs. Par exemple, les LLM ont du mal à compter les caractères d'un mot (c'est un problème numérique, pas un problème de langue) : un évaluateur LLM ne pourra donc pas évaluer correctement ce type de tâche. Il existe parfois des solutions, comme brancher des outils pour les problèmes mal adaptés à un modèle de langage. Mais il est important de connaître ces limites et de faire appel à des évaluateurs humains pour calibrer votre système d'évaluation et établir une référence.

Les évaluateurs LLM fonctionnent bien notamment parce que toutes les informations nécessaires sont dans le contexte d'entrée : le modèle n'a qu'à s'appuyer sur ces informations pour produire le résultat. Quand vous personnalisez des prompts d'évaluation ou construisez vos propres systèmes, gardez cela en tête : ne comptez pas sur les « connaissances internes » du modèle, ni sur un comportement qu'un outil fournirait mieux.

### Gagner en confiance

Pour rendre vos évaluations plus fiables, vous pouvez utiliser des évaluateurs variés : les mêmes prompts et les mêmes réponses, mais évalués par différents modèles, comme Gemini Flash et Pro, voire par différents fournisseurs, comme Gemini, Claude, ChatGPT ou des modèles locaux comme Gemma ou Qwen. C'est la même idée que la répétition des essais pour recueillir plusieurs « avis » et [réduire l'erreur](https://fr.wikipedia.org/wiki/Loi_des_grands_nombres), mais avec des modèles différents, les « avis » sont plus variés.

